# Arena Events IR-Sync Pipeline

Align **block.log** and arena CSV events (trials, bug trajectory, screen touches, etc.) to the Open Ephys timebase using the **IR sync TTL** instead of arena video frame timestamps.

## Why this paradigm?

- Arena **video frame timestamps** (in `frames_timestamps/*.csv`) and the **datetime** in block.log / trials_data / bug_trajectory can disagree (different clocks or drift).
- The **IR sync TTL** is sent to Open Ephys exactly **1 second after** each `ARENA-MAIN - trigger is off` log line (at start and end of recording).
- So we get two anchor points: (log datetime = trigger_off + 1s, OE sample of ir_sync_ttl). We then map any log datetime → ms from OE recording start via a linear mapping.

## Steps

1. Parse **block.log** → table with timestamp, logger, level, message.
2. Find the two **"trigger is off"** lines; add 1 s → reference datetimes for IR sync.
3. In **Open Ephys events.csv**, detect the TTL line with **exactly 2 rising edges** (ir_sync_ttl).
4. Build mapping: log datetime → `ms_axis` (ms from OE recording start).
5. Apply to parsed log and to arena CSVs (trials_data, bug_trajectory, screen_touches, app_events).
6. Optionally save parsed log and aligned CSVs to `analysis/arena_events_ms/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing.arena_log_sync import (
    parse_block_log,
    get_trigger_off_plus_one_second_times,
    detect_ir_sync_ttl_line,
    build_log_datetime_to_oe_ms_mapping,
    log_timestamps_to_oe_ms,
)
from eye_tracking_system_tools.preprocessing.arena_alignment import (
    ARENA_CSV_NAMES,
    ARENA_EVENTS_MS_SUBDIR,
    MS_AXIS,
)

%matplotlib inline

In [ ]:
# Configuration: set block and paths
animal = "PV_208"
date = "2025_12_14"
block_num = "019"

base_path = Path(r"D:\sample_data_for_eye_repo")
block_path = base_path / animal / date / f"block_{block_num}"
arena_videos_dir = block_path / "arena_videos"

# BlockSync gives us oe_path and sample_rate (block_num string e.g. "019")
block = BlockSync(animal, date, block_num, str(base_path))
events_csv_path = block_path / "oe_files" / block.oe_dirname / "events.csv"
sample_rate_hz = float(block.get_sample_rate())

block_log_path = arena_videos_dir / "block.log"

print(f"Block path: {block_path}")
print(f"Arena videos: {arena_videos_dir}")
print(f"Events CSV: {events_csv_path}")
print(f"Sample rate: {sample_rate_hz} Hz")
print(f"Block log: {block_log_path}")

## 1. Parse block.log

In [ ]:
log_df = parse_block_log(block_log_path)
print(f"Parsed {len(log_df)} lines from block.log")
print(log_df.head(15).to_string())

In [ ]:
# Reference datetimes: trigger_off + 1 second (align to ir_sync_ttl in OE)
ref_start_dt, ref_end_dt = get_trigger_off_plus_one_second_times(log_df)
print(f"Reference start (trigger_off + 1s): {ref_start_dt}")
print(f"Reference end   (trigger_off + 1s): {ref_end_dt}")

## 2. Detect IR sync TTL in Open Ephys events

We look for a TTL line with **exactly 2 rising edges**. If none (error shows counts per line), use the fallback: `detect_ir_sync_ttl_line(..., ir_sync_line=N)` to use first and last rising edge of that line.

In [ ]:
# TTL line with exactly 2 rising edges = start and end of arena acquisition.
# If no line has exactly 2, use ir_sync_line=<N> to use first and last rising edge of that line.
ir_sync_line, ir_sync_samples = detect_ir_sync_ttl_line(events_csv_path, expected_rising_edges=2)
# ir_sync_line, ir_sync_samples = detect_ir_sync_ttl_line(events_csv_path, ir_sync_line=4)  # fallback
print(f"IR sync TTL line: {ir_sync_line}")
print(f"IR sync OE samples (rising edges): {ir_sync_samples}")
print(f"IR sync OE times (ms): {ir_sync_samples / (sample_rate_hz / 1000)}")

## 3. Build log datetime → OE ms mapping

In [ ]:
scale, offset = build_log_datetime_to_oe_ms_mapping(
    ref_start_dt, ref_end_dt, ir_sync_samples, sample_rate_hz
)
print(f"Mapping: oe_ms = {scale} * unix_ms + {offset}")

# Sanity: reference times should map to ir_sync OE ms
oe_ms_1 = ir_sync_samples[0] / (sample_rate_hz / 1000)
oe_ms_2 = ir_sync_samples[1] / (sample_rate_hz / 1000)
check_1 = scale * (ref_start_dt.timestamp() * 1000) + offset
check_2 = scale * (ref_end_dt.timestamp() * 1000) + offset
print(f"Check: ref_start -> {check_1:.2f} ms (expected {oe_ms_1:.2f})")
print(f"Check: ref_end   -> {check_2:.2f} ms (expected {oe_ms_2:.2f})")

## 4. Add ms_axis to parsed block.log

In [ ]:
log_df[MS_AXIS] = log_timestamps_to_oe_ms(log_df["timestamp"], scale, offset)
print("Parsed block.log with ms_axis (first and last rows):")
print(log_df[["timestamp", "message", MS_AXIS]].head(5).to_string())
print("...")
print(log_df[["timestamp", "message", MS_AXIS]].tail(5).to_string())

## 5. Align arena CSVs to OE timebase (using IR-sync mapping)

In [ ]:
# We use the same mapping but apply via datetime -> unix_ms -> oe_ms (scale * unix_ms + offset)
# load_and_align_arena_csv uses shift_ms (PC - OE); we have scale/offset instead.
# So we load CSVs and convert time column to oe_ms with our scale/offset.

def align_arena_csv_ir_sync(path: Path, scale: float, offset: float) -> pd.DataFrame:
    df = pd.read_csv(path)
    if df.empty:
        df[MS_AXIS] = pd.Series(dtype=float)
        return df
    if "start_time" in df.columns and "end_time" in df.columns:
        df["ms_axis_start"] = log_timestamps_to_oe_ms(df["start_time"], scale, offset)
        df["ms_axis_end"] = log_timestamps_to_oe_ms(df["end_time"], scale, offset)
    elif "time" in df.columns:
        df[MS_AXIS] = log_timestamps_to_oe_ms(df["time"], scale, offset)
    else:
        df[MS_AXIS] = np.nan
    return df

aligned = {}
for name in ARENA_CSV_NAMES:
    path = arena_videos_dir / name
    if not path.exists():
        continue
    key = path.stem
    aligned[key] = align_arena_csv_ir_sync(path, scale, offset)
    print(f"{key}: {len(aligned[key])} rows, columns: {list(aligned[key].columns)}")

In [ ]:
if "trials_data" in aligned:
    print("trials_data (first 3 rows):")
    print(aligned["trials_data"][["start_time", "ms_axis_start", "ms_axis_end"]].head(3))
if "bug_trajectory" in aligned:
    print("\nbug_trajectory (first 3 rows):")
    print(aligned["bug_trajectory"][["time", MS_AXIS]].head(3))

## 6. Save parsed log and aligned CSVs (optional)

In [ ]:
analysis_path = block_path / "analysis"
out_dir = analysis_path / ARENA_EVENTS_MS_SUBDIR
out_dir.mkdir(parents=True, exist_ok=True)

# Save parsed block log with ms_axis
log_out = out_dir / "block_log_parsed_ms_axis.csv"
log_df.to_csv(log_out, index=False)
print(f"Saved parsed log: {log_out}")

# Save aligned arena CSVs (same names as arena_alignment: stem_ms_axis.csv)
for key, df in aligned.items():
    out_name = f"{key}_ms_axis.csv"
    out_path = out_dir / out_name
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

# Save mapping constants for reproducibility
import json
mapping_info = {
    "ref_start_dt": str(ref_start_dt),
    "ref_end_dt": str(ref_end_dt),
    "ir_sync_line": ir_sync_line,
    "ir_sync_samples": ir_sync_samples.tolist(),
    "sample_rate_hz": sample_rate_hz,
    "scale": scale,
    "offset": offset,
}
mapping_path = out_dir / "arena_ir_sync_mapping.json"
mapping_path.write_text(json.dumps(mapping_info, indent=2))
print(f"Saved mapping info: {mapping_path}")